In [4]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga del shapefile de secciones censales

In [5]:
# Cargo el GDF con las secciones censales
ruta_shp_secciones = os.path.join(DATA_OUTPUTS_DIR, "Shapefiles", "cyl_2022.shp")
gdf_secciones = gpd.read_file(ruta_shp_secciones)

# Compruebo que se haya cargado
print(f"Vista del gdf de secciones:\n{gdf_secciones.head()}")

Vista del gdf de secciones:
        CUSEC                      NMUN   NPRO  Seccion_id  \
0  0500101001                   Adanero  Ávila           1   
1  0500201001                Adrada, La  Ávila           2   
2  0500201002                Adrada, La  Ávila           3   
3  0500501001                  Albornos  Ávila           4   
4  0500701001  Aldeanueva de Santa Cruz  Ávila           5   

                                            geometry  
0  POLYGON ((365705.918 4536187.034, 365958.915 4...  
1  POLYGON ((363065.743 4462346.46, 363062.106 44...  
2  POLYGON ((361529.181 4469725.932, 361631.182 4...  
3  POLYGON ((343504.663 4523882.125, 343549.661 4...  
4  POLYGON ((294940.799 4473589.074, 294982.799 4...  


# Carga del fichero de servicios sociales

El fichero de servicios sociales se extrae de datosabiertos.jcyl disponible en la siguiente dirección: 

https://datosabiertos.jcyl.es/web/jcyl/set/es/sociedad-bienestar/servicios_sociales/1284208630291

In [6]:
# Cargo el fichero csv con los centros docentes
ruta_csv_servicios_sociales = os.path.join(DATA_INPUTS_DA, "servicios-de-caracter-social.csv")
df_servicios_sociales = pd.read_csv(ruta_csv_servicios_sociales, sep=";", encoding="utf-8")
df_servicios_sociales["servicio_id"] = range(1, len(df_servicios_sociales) + 1)

# Veo una muestra de su estructura y contenido
print(df_servicios_sociales.info())
df_servicios_sociales.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8231 entries, 0 to 8230
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Denominación               8229 non-null   object
 1   Fecha última modificación  8229 non-null   object
 2   Dirección                  8230 non-null   object
 3   Código Provincia           8231 non-null   object
 4   Codigo Municipio INE       8231 non-null   object
 5   Provincia                  8231 non-null   object
 6   Localidad                  8231 non-null   object
 7   CP                         8229 non-null   object
 8   Zona Acción Social         8229 non-null   object
 9   Teléfono                   7279 non-null   object
 10  Fax                        643 non-null    object
 11  email                      3170 non-null   object
 12  Web                        237 non-null    object
 13  Entidad_Titular            8226 non-null   object
 14  Naturale

,Denominación,Fecha última modificación,Dirección,Código Provincia,Codigo Municipio INE,Provincia,Localidad,CP,Zona Acción Social,Teléfono,Fax,email,Web,Entidad_Titular,Naturaleza,Sector,Tipo,Subtipo,servicio_id
4320,ESCOLARIZACION DE NIÑOS DE FAMILIAS NECESITADAS,2022-10-24,CALLE MADRE DE DIOS 9,47,47186,VALLADOLID,VALLADOLID,47004,VALLADOLID - HOSPITAL,983 352640,NaN,NaN,NaN,VOLUNTARIADO VICENCIANO DE VALLADOLID,ASOCIACIÓN,INFANCIA (Y JUVENTUD),Servicio,Prevención,4321
6541,ATENCION A PERSONAS MAYORES,2007-05-03,AVENIDA RECUNCO (EL) S/N,24,24119,LEON,PRIARANZA DEL BIERZO,24448,PUENTE DOMINGO FLOREZ,NaN,NaN,NaN,NaN,"ASOCIACION DE PENSIONISTAS Y JUBILADOS ""VALLE ...",ASOCIACIÓN,PERSONAS MAYORES,"Residencia, Servicio",Sociocultural,6542
6702,"SERVICIO DE PROMOCIÓN, MANTENIMIENTO Y RECUPER...",2022-07-25,"POLIGONO INDUSTRIAL, Nº 4",37,37376,SALAMANCA,VITIGUDINO,37210,VITIGUDINO,923 186893,923 186860,asprodes@asprodes.es,www.asprodes.es,ASPRODES - FEAPS SALAMANCA,ASOCIACIÓN,POBLACIÓN GENERAL,Servicio,"Promoción de la autonomía personal: Promoción,...",6703
1724,OCIO Y TIEMPO LIBRE CON MENORES,2007-05-03,CALLE CRUZ ROJA S/N,37,37274,SALAMANCA,SALAMANCA,37007,SALAMANCA - ZONA 7 - SAN BERNARDO - VIDAL,923 221032,NaN,NaN,NaN,CRUZ ROJA ESPAÑOLA,ASOCIACIÓN,INFANCIA (Y JUVENTUD),Servicio,Prevención,1725
5763,"PROMOCION, MANTENIMIENTO Y RECUPERACION DE LA ...",2021-09-30,PLAZA SAN JUAN 7,47,47186,VALLADOLID,VALLADOLID,47002,VALLADOLID - CAMPILLO,983 398661,NaN,info@villasaluten.com,NaN,PSM SALUD S.L.,SOCIEDADES,POBLACIÓN GENERAL,Servicio,"Promoción de la autonomía personal: Promoción,...",5764


# Filtrado del fichero

## Subtipo
Actualmente la cartera de servicios publicados incluye servicios catalogados y no catalogados. Los catalogados son los que tienen en la columna
subtipo un valor (X.XX), por lo que únicamente se seleccionarán estos servicios, que son los que tienen una definicion formal según la Junta de 
Castilla y León. Se define también una función que permite extraer el código del servicio y su descripción.

## CEAS
Este catálogo de servicios incluye también algunos CEAS, que se excluirán porque se tratarán en otro cuaderno de trabajo.

## SECTOR
Contiene varios sectores relacionados con la inclusión social, por lo que es mejor agrupar para facilitar el modelado.

In [7]:
# Saco servicios catalogados (oficiales) vs no catalogados, y excluyo posibles CEAS que se tratarán a parte
patron = r"\(\s*\d+\.\d+\s*\)"

# Catalogados
df_servicios_sociales_catalogados = df_servicios_sociales[
    (df_servicios_sociales["Subtipo"].astype(str).str.contains(patron, regex=True)) &
    (~df_servicios_sociales["Denominación"].astype(str).str.startswith("CEAS"))
].copy()


print(f"✅ Catalogados: {len(df_servicios_sociales_catalogados)}")

def extraer_codigo_catalogo(df, columna_subtipo="Subtipo"):
    """
    Extrae el código de servicio (formato N.DD) y la descripción del servicio
    de la columna 'Subtipo', generando las columnas:
      - 'Codigo_catalogo': código del servicio (N.DD)
      - 'Des_servicio': descripción limpia del servicio
    """

    df = df.copy()

    # Patrón para capturar el número (N.DD) entre paréntesis con espacios opcionales
    patron_codigo = r"\(\s*(\d+\.\d+)\s*\)"
    
    # 1️⃣ Extraer código
    df["cod_catalogo"] = (
        df[columna_subtipo]
        .astype(str)
        .str.extract(patron_codigo)[0]
        .str.strip()
    )

    # 2️⃣ Extraer descripción eliminando el código
    df["des_catalogo"] = (
        df[columna_subtipo]
        .astype(str)
        .str.replace(patron_codigo, "", regex=True)
        .str.strip(" -:")  # limpia espacios, guiones y dos puntos residuales
        .str.replace(r"\s+", " ", regex=True)  # normaliza espacios intermedios
    )

    # 3️⃣ Verificación
    n_codigos = df["cod_catalogo"].notna().sum()
    print(f"✅ Códigos de catálogo extraídos correctamente ({n_codigos} registros con código detectado).")

    return df

df_servicios_sociales_catalogados = extraer_codigo_catalogo(df_servicios_sociales_catalogados)

print(f"Muestra de servicios catalogados:\n{df_servicios_sociales_catalogados[['cod_catalogo', 'des_catalogo']].sample(5)}\n")

# Muestro conteo por sector
print(f"Sectores disponibles:\n{df_servicios_sociales_catalogados['Sector'].value_counts()}\n")

# Agrupo por inclusión social
df_servicios_sociales_catalogados.loc[
    df_servicios_sociales_catalogados["Sector"].str.startswith("INCLUSIÓN", na=False),
    "Sector"
] = "INCLUSIÓN SOCIAL"

print(f"Sectores disponibles tras agrupar:\n{df_servicios_sociales_catalogados['Sector'].value_counts()}\n")

✅ Catalogados: 1325
✅ Códigos de catálogo extraídos correctamente (1325 registros con código detectado).
Muestra de servicios catalogados:
     cod_catalogo                                       des_catalogo
6456         27.5  Serv. apoyo para la integración sociolaboral d...
30            2.6  Servicio general de información, orientación, ...
1185         27.5  Serv. apoyo para la integración sociolaboral d...
4257        26.16         Servicio de apoyo para la inclusión social
380         26.16         Servicio de apoyo para la inclusión social

Sectores disponibles:
Sector
INCLUSIÓN SOCIAL             422
PERSONAS MAYORES             176
PERSONAS CON DISCAPACIDAD    157
POBLACIÓN GENERAL            129
TOXICOMANÍAS Y ADICCIONES    105
MUJER                         96
INFANCIA (Y JUVENTUD)         89
(FAMILIA)                     73
INCLUSIÓN SOCIAL-[TR]         46
VOLUNTARIADO                  22
INCLUSIÓN SOCIAL-[RE]         10
Name: count, dtype: int64

Sectores disponibles tras a

In [8]:
# Filtro los campos que quiero
cols_to_keep = ["Dirección", "Localidad", "Teléfono", "email", "Entidad_Titular","Sector","servicio_id", "CP", "cod_catalogo", "des_catalogo"]
df_servicios_sociales_catalogados = df_servicios_sociales_catalogados[cols_to_keep]


# Asignación de código de sección censal (CUSEC)
Como no se dispone por defecto del CUSEC pero sí de del CP, y se ha definido una funcion que obtiene el CUSEC del CP,
aprovechamos para asignarlo

In [9]:
df_servicios_sociales_catalogados = asignar_cusec_por_cp(df_servicios_sociales_catalogados, columna_cp = "CP")
df_servicios_sociales_catalogados.sample(5)

# Borro despues la columna CP que no me interesa
df_servicios_sociales_catalogados = df_servicios_sociales_catalogados.drop(columns=["CP"])


✔️ 1325 registros procesados
   🔹 CUSEC completados: 1325 | Sin asignar: 0
   🔹 Seccion_id completados: 1293 | Sin asignar: 32


# Cálculo de distancia mínima a un servicio desde un CUSEC y accesibilidad de servicios para un CUSEC
Es necesario para cada sección censal calcular la distancia mínima existente hacia un servicio de un tipo, así como crear
una medida de accesibilidad de un servicio concreto desde una sección cens

In [10]:
# Recorremos cada categoría educativa
df_resultado = pd.DataFrame({"CUSEC": gdf_secciones["CUSEC"].unique()})

# Tabla global de relaciones
lista_relaciones = []

for sector in df_servicios_sociales_catalogados["Sector"].dropna().unique():
    print(f"\n➡️ Procesando sector: {sector}")

    # Filtrar centros de la categoría
    df_sector = df_servicios_sociales_catalogados[df_servicios_sociales_catalogados["Sector"] == sector].copy()

    # Normalizar nombre para usarlo como prefijo de columnas
    nombre_norm = (
        sector.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
    )

    # Calcular accesibilidad (versión v2 o v3)
    df_acc, df_rel = calcular_accesibilidad_v2(
        df_servicio=df_sector,
        nombre_servicio=nombre_norm,
        col_id="servicio_id"              # ID del centro educativo
    )

    lista_relaciones.append(df_rel)

    # Para no tener problemas con la sección
    if "Seccion_id" in df_acc.columns:
        df_acc = df_acc.drop(columns=["Seccion_id"])
        
    # Unir al DF principal
    df_resultado = df_resultado.merge(df_acc, on="CUSEC", how="outer")

df_relaciones_global = pd.concat(lista_relaciones, ignore_index=True)

df_resultado.head()


➡️ Procesando sector: INCLUSIÓN SOCIAL


Accesibilidad inclusión_social: 100%|████████████████████████████████████████| 3535/3535 [00:15<00:00, 222.01sección/s]



➡️ Procesando sector: POBLACIÓN GENERAL


Accesibilidad población_general: 100%|███████████████████████████████████████| 3535/3535 [00:08<00:00, 436.94sección/s]



➡️ Procesando sector: TOXICOMANÍAS Y ADICCIONES


...


➡️ Procesando sector: VOLUNTARIADO


Accesibilidad voluntariado: 100%|████████████████████████████████████████████| 3535/3535 [00:05<00:00, 604.70sección/s]



➡️ Procesando sector: PERSONAS MAYORES


Accesibilidad personas_mayores: 100%|████████████████████████████████████████| 3535/3535 [00:09<00:00, 389.59sección/s]


,CUSEC,dist_min_inclusión_social_km,n_inclusión_social_1km,n_inclusión_social_5km,n_inclusión_social_15km,n_inclusión_social_30km,disp_ponderada_inclusión_social,dist_min_población_general_km,n_población_general_1km,n_población_general_5km,...,n_voluntariado_5km,n_voluntariado_15km,n_voluntariado_30km,disp_ponderada_voluntariado,dist_min_personas_mayores_km,n_personas_mayores_1km,n_personas_mayores_5km,n_personas_mayores_15km,n_personas_mayores_30km,disp_ponderada_personas_mayores
0,0500101001,26.456992,0,0,0,1,0.1,30.426842,0,0,...,0,0,0,0.0,16.637958,0,0,0,2,0.2
1,0500201001,9.932053,0,0,1,1,0.3,41.154148,0,0,...,0,0,0,0.0,6.209785,0,0,2,3,0.7
2,0500201002,8.919316,0,0,1,9,1.1,34.793629,0,0,...,0,0,0,0.0,4.415298,0,1,2,4,1.1
3,0500501001,19.273638,0,0,0,31,3.1,19.273638,0,0,...,0,0,1,0.1,11.828076,0,0,1,5,0.7
4,0500701001,9.204836,0,0,3,6,1.2,67.397336,0,0,...,0,0,0,0.0,22.063578,0,0,0,2,0.2


In [11]:
df_resultado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3535 entries, 0 to 3534
Data columns (total 55 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   CUSEC                                     3535 non-null   object 
 1   dist_min_inclusión_social_km              3535 non-null   float64
 2   n_inclusión_social_1km                    3535 non-null   int64  
 3   n_inclusión_social_5km                    3535 non-null   int64  
 4   n_inclusión_social_15km                   3535 non-null   int64  
 5   n_inclusión_social_30km                   3535 non-null   int64  
 6   disp_ponderada_inclusión_social           3535 non-null   float64
 7   dist_min_población_general_km             3535 non-null   float64
 8   n_población_general_1km                   3535 non-null   int64  
 9   n_población_general_5km                   3535 non-null   int64  
 10  n_población_general_15km            

In [16]:
# Integro el CUSEC_ID en el de relaciones
print(df_relaciones_global.sample(5))

df_relaciones_global = (
    df_relaciones_global
    .merge(
        gdf_secciones.drop(columns=["geometry","NMUN","NPRO"]),
        left_on="CUSEC_origen",
        right_on="CUSEC",
        how="left"
    )
)

       CUSEC_origen  servicio_id CUSEC_servicio     distancia
111299   4705201001         2305     4718604040   5701.920579
183499   4911501001          645     4927502006  29907.676065
254273   4718607020         6832     4718610001   2431.234955
215211   4705201005         4235     4718604040   3152.512240
42139    3727401010         3260     3727403009   1735.166677


,CUSEC_origen,servicio_id,CUSEC_servicio,distancia,CUSEC,Seccion_id
106265,3712901001,1070,3727403009,9649.512443,3712901001,1710
6575,0905902005,5440,0990701001,7758.577318,0905902005,387
11320,0905906011,6350,0905905035,5004.412436,0905906011,465
55014,4013601001,2792,4019407004,28410.165891,4013601001,2233
143022,4927502017,4403,4927503004,5487.311719,4927502017,3453


In [17]:
# Realmente solo me interesan las columnas 'Seccion_id, distancia y servicio_id'
df_relaciones_global = df_relaciones_global.drop(
    columns=["CUSEC_origen", "CUSEC_servicio", "CUSEC"]
)

In [18]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DA, exist_ok=True)

# Rutas de salida
ruta_resultado = os.path.join(DATA_OUTPUTS_DA, "accesibilidad_servicios_sociales.csv")
ruta_centros = os.path.join(DATA_OUTPUTS_DA, "servicios_sociales_final.csv")
ruta_relaciones = os.path.join(DATA_OUTPUTS_DA, "relaciones_servicios.csv")

# Guardar DataFrames
df_resultado.to_csv(ruta_resultado, index=False, encoding="utf-8-sig")
df_servicios_sociales_catalogados.to_csv(ruta_centros, index=False, encoding="utf-8-sig")

df_servicios_sociales_catalogados.to_csv(
    ruta_centros,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

df_relaciones_global.to_csv(
    ruta_relaciones,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivos guardados correctamente en:\n- {DATA_OUTPUTS_DA}")

✅ Archivos guardados correctamente en:
- D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DA_Dim_servicios
